## ML_1033: Multi-Head Cross-Attention

**Difficulty**: Medium | **TorchCode**: #23

### Problem
Implement multi-head cross-attention for encoder-decoder architectures. Queries come from the decoder (`x_q`), while Keys and Values come from the encoder (`x_kv`). No causal mask is applied.

### Formula
$$Q = x_q W^Q, \quad K = x_{kv} W^K, \quad V = x_{kv} W^V$$
$$\text{out} = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V \cdot W^O$$

In [ ]:
import torch
import torch.nn as nn
import math

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv):
        B, S_q, _ = x_q.shape
        S_kv = x_kv.shape[1]
        q = self.W_q(x_q).view(B, S_q,  self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x_kv).view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x_kv).view(B, S_kv, self.num_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        weights = torch.softmax(scores, dim=-1)
        attn = torch.matmul(weights, v)
        return self.W_o(attn.transpose(1, 2).contiguous().view(B, S_q, -1))

In [ ]:
import torch
import torch.nn as nn

# --- Test 1: Output shape ---
attn = MultiHeadCrossAttention(d_model=64, num_heads=4)
assert isinstance(attn, nn.Module)
assert attn(torch.randn(2, 6, 64), torch.randn(2, 10, 64)).shape == (2, 6, 64)

# --- Test 2: Different Q and KV sequence lengths ---
attn2 = MultiHeadCrossAttention(d_model=32, num_heads=2)
assert attn2(torch.randn(1, 3, 32), torch.randn(1, 20, 32)).shape == (1, 3, 32)

# --- Test 3: No causal mask — changing last KV affects all Q positions ---
torch.manual_seed(0)
attn3 = MultiHeadCrossAttention(d_model=32, num_heads=2)
x_q = torch.randn(1, 4, 32)
x_kv = torch.randn(1, 6, 32)
out1 = attn3(x_q, x_kv)
x_kv2 = x_kv.clone(); x_kv2[:, -1] = torch.randn(1, 32)
out2 = attn3(x_q, x_kv2)
assert not torch.allclose(out1[:, 0], out2[:, 0], atol=1e-5)

# --- Test 4: Gradient flow ---
x_q = torch.randn(1, 4, 32, requires_grad=True)
x_kv = torch.randn(1, 6, 32, requires_grad=True)
attn3(x_q, x_kv).sum().backward()
assert x_q.grad is not None and x_kv.grad is not None

print('All tests passed!')